In [1]:
import datetime
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import json

from sklearn import datasets, ensemble, model_selection
from scipy.stats import anderson_ksamp


In [2]:
content = requests.get("https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip").content
with zipfile.ZipFile(io.BytesIO(content)) as arc:
    raw_data = pd.read_csv(arc.open("hour.csv"), header=0, sep=',', parse_dates=['dteday'])

In [3]:
raw_data.index = raw_data.apply(lambda row: datetime.datetime.combine(row.dteday.date(), datetime.time(row.hr)),
                                axis=1)

In [4]:
raw_data.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
2011-01-01 00:00:00,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
2011-01-01 01:00:00,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2011-01-01 02:00:00,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
2011-01-01 03:00:00,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
2011-01-01 04:00:00,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [5]:
from scipy import stats
import random


#Significance level

significance_level = 0.05
rejected = 0

numerical_features = ['temp', 'atemp', 'hum', 'windspeed', 'mnth', 'hr', 'weekday']
categorical_features = ['season', 'holiday', 'workingday']



reference = raw_data.loc['2011-01-01 00:00:00':'2011-01-28 23:00:00']
current = raw_data.loc['2011-01-29 00:00:00':'2011-02-28 23:00:00']


for col in numerical_features:
    test = stats.ks_2samp(reference[col], current[col])
    
    #print(col, test[1])
    if test[1] < significance_level:
        rejected += 1
        #print("Column rejected", col)
    else:
        print("Columns accepted ",col)

print("We rejected ",rejected," columns in total out of {} columns".format(len(numerical_features)))


Columns accepted  hr
Columns accepted  weekday
We rejected  5  columns in total out of 7 columns


In [6]:
from scipy.stats import chi2_contingency

def drift_chisquare(sample1, sample2):
    return chi2_contingency([sample1, sample2])[1]

In [7]:
for col in categorical_features:
    val = drift_chisquare(reference[col].value_counts(),current[col].value_counts() )

    print(col,val)
    rejected = 0
    if val < significance_level:
        rejected += 1
        print("Column rejected", col)

print("We rejected ",rejected," columns in total out of {} columns".format(len(categorical_features)))

season 1.0
holiday 0.6986573626612528
workingday 0.5917879941201512
We rejected  0  columns in total out of 3 columns


### Model performance

In [8]:
target = 'cnt'
prediction = 'prediction'
numerical_features = ['temp', 'atemp', 'hum', 'windspeed', 'mnth', 'hr', 'weekday']
categorical_features = ['season', 'holiday', 'workingday', ]#'weathersit']

In [9]:
reference = raw_data.loc['2011-01-01 00:00:00':'2011-01-28 23:00:00']
current = raw_data.loc['2011-01-29 00:00:00':'2011-02-28 23:00:00']

In [10]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    reference[numerical_features + categorical_features],
    reference[target],
    test_size=0.3
)

In [11]:
regressor = ensemble.RandomForestRegressor(random_state = 0)
regressor.fit(X_train, y_train)

RandomForestRegressor(random_state=0)

In [12]:
preds_test = regressor.predict(X_test)

In [13]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
ref_mae=mean_absolute_error(y_test,preds_test)
ref_mse=mean_squared_error(y_test,preds_test)

print("MAE",ref_mae)
print("MSE",ref_mse)


MAE 10.333387096774194
MSE 222.1941016129032


In [14]:
current_x=current[numerical_features + categorical_features]
current_y=current[target]

current_pred = regressor.predict(current_x)

In [15]:
print("MAE",mean_absolute_error(current_y,current_pred))
print("MSE",mean_squared_error(current_y,current_pred))

MAE 20.090431154381086
MSE 1069.2556317107094


In [16]:
import mlflow
from mlflow.tracking import MlflowClient
import os

In [17]:
mlflow.set_experiment("Bicycle–Sharing")

<Experiment: artifact_location='file:///d:/Scaler/Scaler_MLOps/code/Class_code/9_ML_System_design_1/mlruns/209720353589099245', creation_time=1773076405919, experiment_id='209720353589099245', last_update_time=1773076405919, lifecycle_stage='active', name='Bicycle–Sharing', tags={}>

In [18]:
with mlflow.start_run():

    mlflow.set_tag('mlflow.runName','Refrence_run')
    mlflow.log_metric("MAE",ref_mae)
    mlflow.log_metric("MSE",ref_mse)
    

    mlflow.sklearn.log_model(regressor, "model")

2026/03/09 22:45:36 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


In [20]:
experiment_batches = [
    ('2011-01-29 00:00:00','2011-02-07 23:00:00'),
    ('2011-02-08 00:00:00','2011-02-14 23:00:00'),
    ('2011-02-15 00:00:00','2011-02-21 23:00:00'),
]

In [21]:
#start new run
for date in experiment_batches:
    with mlflow.start_run() as run: #inside brackets run_name='test'

        mlflow.set_tag('mlflow.runName',"run_"+str(date[0])+" : "+str(date[1]))
        # Get metrics
        current_data=current.loc[date[0]:date[1]]
        current_x=current_data[numerical_features + categorical_features]
        current_y=current_data[target]
        current_pred = regressor.predict(current_x)

        mae=mean_absolute_error(current_y,current_pred)
        mse=mean_squared_error(current_y,current_pred)

        mlflow.log_metric('MAE', round(mae, 3))
        mlflow.log_metric('MSE', round(mse, 3))
